<a href="https://colab.research.google.com/github/wetherc/data-2000/blob/sp26/labs/083_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NOTE**: I suggest running this notebook on an A100 GPU with high RAM

# Lab Exercise: Building a Neural Chatbot from Reddit Data

Welcome to this interactive lab! Today, we will explore Natural Language Processing (NLP) by building a simple text-generation neural network (a foundational concept for chatbots).

We will use a subset of the **Reddit** dataset from TensorFlow Datasets. Our goal is to train a Recurrent Neural Network (RNN) to understand the structure of the text and generate plausible responses based on a conversation history.

### Learning Objectives:
1.  **Data Loading & Preprocessing**: Learn how to handle raw text data, tokenize it, and prepare sequences for a neural network.
2.  **Model Building**: Understand the architecture of an LSTM (Long Short-Term Memory) network for sequence prediction.
3.  **Training & Validation**: Train the model and monitor its loss.
4.  **Interactive Inference**: Build a loop to converse with your trained model, maintaining conversation context.

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import re
import time
import gdown



NUM_SAMPLES = 250000
VOCAB_SIZE = 50000
SEQ_LENGTH = 100  # the length of the context window
BATCH_SIZE = 64
EPOCHS = 10

In [ ]:
import nltk
nltk.download('punkt', quiet=False)
nltk.download('punkt_tab', quiet=False)

from nltk.tokenize import sent_tokenize

## Part 1: Data Loading


We will download the `reddit` dataset (via [Tensorflow Datasets](https://www.tensorflow.org/datasets/catalog/reddit)). This dataset consists of 3,848,330 posts with an average length of 270 words for content, and 28 words for the summary. We will use the document text to train our language model. Because of the large dataset size, we will train our model on only a sample of 250,000 of these. You can adjust the size of the training dataset by altering `NUM_SAMPLES` above.

The dataset is about 18GB decompressed, so it'll take around 15 minutes to download and extract the entire thing. For the lab exercise, I'd recomment **NOT** running this cell, and instead skipping down to where I load in a smaller, 250,000-record subset from the course's Google Drive

In [ ]:
# dataset, info = tfds.load(
#     "reddit",
#     split=f"train[:{NUM_SAMPLES}]",
#     with_info=True)

In [ ]:
# from google.colab import drive
# import os

# drive.mount("/content/drive")

# local_dir = "reddit_subset_dataset"
# zip_file = "reddit_subset_dataset.zip"
# drive_dir = "/content/drive/My Drive/DATA2000/datasets/"
# drive_path = os.path.join(drive_dir, zip_file)

# os.makedirs(drive_dir, exist_ok=True)
# dataset.save(local_dir)

# !zip -q -r {zip_file} {local_dir}
# !cp {zip_file} "{drive_path}"

# print("Dataset zipped and saved to Drive successfully!")

In [ ]:
file_id = "1Sd_ketpDsUTamhGO2X6ZWUT1xR0W7C7p"
url = f"https://drive.google.com/uc?id={file_id}"
output_zip = "/content/reddit_subset_dataset.zip"
extracted_dir = "/content/reddit_subset_dataset"

In [ ]:
print("Downloading dataset from Google Drive...")
gdown.download(url, output_zip, quiet=False)

print("Extracting dataset...")
!unzip -q {output_zip} -d /content/

In [ ]:
print("Loading dataset...")
dataset = tf.data.Dataset.load(extracted_dir)
print("Dataset loaded successfully!")

In [ ]:
texts = []
_counter = 0
for item in dataset.take(NUM_SAMPLES):
    # The "reddit" dataset uses "content" for the main post body
    text = item["content"].numpy().decode("utf-8")
    text = re.sub(r"\s+", " ", text).strip()

    # Split into sentences and add '<EOS>' to each
    sentences = sent_tokenize(text)
    processed_text = " ".join([s + " <EOS>" for s in sentences])
    texts.append(processed_text)
    _counter +=1

    if _counter % 25000 == 0:
        print(f"Processed {_counter} / {NUM_SAMPLES} records")

In [ ]:
print(f"Loaded {len(texts)} posts. Here is a sample:")
print("---")
for idx in range(0, len(texts[0]), 40):
    print(texts[0][idx:min(idx + 40, len(texts[0]))])

## Part 2: Data Preprocessing

We will process the raw text strings by:
1. **Custom Standardization**: Real-world data (like Reddit) contains noise such as URLs and non-standard punctuation. We will define a custom standardization function to clean this data at the graph level within TensorFlow.
2. **Tokenization via `TextVectorization`**: We map the most frequent words to integer IDs. The vocabulary size bounds our classification space (and memory footprint). Words outside this vocabulary become Out-Of-Vocabulary (OOV) tokens.
3. **Autoregressive Sequencing**: For language modeling, we train the model to predict $P(w_t | w_{1:t-1})$. We split our tokenized corpus into sequences of `SEQ_LENGTH + 1`, using the first `SEQ_LENGTH` tokens as the input and the sequence shifted by one timestep as the target.

In [ ]:
import string

# Combine all text into one corpus
text_corpus = " ".join(texts)

@tf.keras.utils.register_keras_serializable()
def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    stripped_urls = tf.strings.regex_replace(lowercase, r"http\S+", "")
    stripped_punct = tf.strings.regex_replace(
        stripped_urls,
        f"[{re.escape(string.punctuation)}]",
        "")
    return stripped_punct

# Create the Vectorizer
vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=SEQ_LENGTH + 1,
    standardize=custom_standardization
)

print("Adapting vectorizer to the corpus (this might take a moment)...")
vectorize_layer.adapt(texts)

# Get the vocabulary to map IDs back to words later
vocab = vectorize_layer.get_vocabulary()
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for idx, word in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)}")

In [ ]:
# To train a language model, we slice the text into sequences of SEQ_LENGTH + 1
# Example:  [word1, word2, word3, word4]
#   Input:  [word1, word2, word3]
#  Target:  [word2, word3, word4]

def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

vectorized_texts = vectorize_layer(texts)

dataset_seq = tf.data.Dataset.from_tensor_slices(vectorized_texts)
dataset_processed = dataset_seq.map(split_input_target)

dataset_processed = (
    dataset_processed
    .shuffle(10000)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

## Part 3: Advanced Model Architecture (Stacked LSTM)

Instead of a vanilla, single-layer RNN, we will deploy a Stacked LSTM (Long Short-Term Memory) architecture. Vanilla RNNs suffer heavily from the vanishing gradient problem when computing Backpropagation Through Time (BPTT).

### The Vanishing Gradient Problem


When training an RNN on long sequences, the network calculates gradients to update its weights. Because these gradients are calculated using the chain rule across many time steps, the repeated multiplication of small numbers (derived from activation functions like tanh or sigmoid) causes the overall gradient to shrink exponentially. This "vanishing" gradient means the network cannot effectively update weights to learn long-term dependencies; it "forgets" earlier parts of the sequence.


### How LSTMs Solve This (The 3 Gates)


LSTMs mitigate vanishing gradients by utilizing a constant-error carousel called the cell state ($C_t$), which acts like a conveyor belt carrying information straight down the entire sequence with only minor linear interactions. This flow is regulated by three distinct gates, each typically composed of a sigmoid neural net layer and a pointwise multiplication operation:
1. **Forget Gate**: Decides what information from the previous cell state should be thrown away or kept.
2. **Input Gate**: Decides which new information from the current input and previous hidden state will be stored in the cell state.
3. **Output Gate**: Determines what the next hidden state should be, outputting a filtered version of the updated cell state.

By stacking multiple LSTMs, we allow the network to learn hierarchical representations of the text (e.g., lower layers might learn syntax, while higher layers capture semantic meaning). We also introduce dropout (here specified as a parameter to the LSTM cells, rather than as a standalone layer that we insert ourselves), which you'll remember are a regularization technique that randomly zeros out a fraction of connections during training to prevent the network from overfitting.

In [ ]:
EMBEDDING_DIM = 256
RNN_UNITS = 384
NUM_LAYERS = 2
DROPOUT_RATE = 0.2

class ChatbotModel(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, rnn_units, num_layers, dropout_rate):
        super().__init__()
        # The embedding layer maps integer IDs to dense semantic vectors
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

        # Stack multiple LSTM layers
        self.lstm_layers = []
        for _ in range(num_layers):
            self.lstm_layers.append(
                tf.keras.layers.LSTM(
                    rnn_units,
                    return_sequences=True,
                    return_state=True,
                    dropout=dropout_rate
                )
            )

        # Final linear projection back to vocabulary size
        self.dense = tf.keras.layers.Dense(vocab_size)

    def call(self, inputs, states=None, return_state=False, training=False):
        x = self.embedding(inputs, training=training)

        if states is None:
            # Initialize a list of states for each layer
            states = [None] * len(self.lstm_layers)

        new_states = []
        for i, lstm in enumerate(self.lstm_layers):
            x, state_h, state_c = lstm(x, initial_state=states[i], training=training)
            new_states.append([state_h, state_c])

        x = self.dense(x, training=training)

        if return_state:
            return x, new_states
        else:
            return x

In [ ]:
model = ChatbotModel(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    rnn_units=RNN_UNITS,
    num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE
)

In [ ]:
# Build the model by passing a dummy batch
for input_example_batch, target_example_batch in dataset_processed.take(1):
    example_batch_predictions = model(input_example_batch)
    print(f"Prediction shape: {example_batch_predictions.shape} # (batch_size, sequence_length, vocab_size)")

In [ ]:
model.summary()

## Part 4: Training and Cross-Entropy


We compile the model using Sparse Categorical Crossentropy. In the context of language modeling, minimizing cross-entropy is equivalent to maximizing the log-likelihood of the true next word. This relates directly to Perplexity, a standard evaluation metric for language models ($Perplexity = e^{CrossEntropy}$). Lower loss means our model assigns a higher probability to the actual sequence of human-written text.

**NOTE**: This model takes about 30 minutes to train on an A100, so I will be using a previously trained version rather than letting it train live in class. You're welcome to run this code yourself and train your own model, however

In [ ]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer="adam", loss=loss)

In [ ]:
# print(f"Starting training for {EPOCHS} epochs...")
# history = model.fit(dataset_processed, epochs=EPOCHS)

In [ ]:
# import os

# local_model = "reddit_lstm_model.weights.h5"
# drive_model_dir = "/content/drive/My Drive/DATA2000/models/"
# drive_model_path = os.path.join(drive_model_dir, local_model)

# os.makedirs(drive_model_dir, exist_ok=True)

In [ ]:
# print("Saving model weights locally...")
# model.save_weights(local_model)

# print("Copying model to Google Drive...")
# !cp {local_model} "{drive_model_path}"
# print(f"Model weights saved to Drive at {drive_model_path} successfully!")

In [ ]:
model_file_id = "101_HHig9SFYEJogBVGIxrMwD__wIIRZD"
model_url = f"https://drive.google.com/uc?id={model_file_id}"
model_output = "/content/reddit_lstm_model.weights.h5"

gdown.download(model_url, model_output, quiet=False)

In [ ]:
model.build(input_shape=(None, SEQ_LENGTH))
model.load_weights(model_output)
print("Model weights loaded successfully!")

## Part 5: The Interactive Chatbot
Now that the model is trained, it can predict the next word. We will build a helper function to generate text word-by-word, feeding its own predictions back into itself.

We will also maintain a context window of the last 12 turns (or 12 tokens/phrases) so the bot has memory of the immediate conversation history.

In [ ]:
def generate_response(model, context_text, num_generate=20, temperature=1.0):
    # Convert the context text into token IDs
    input_ids = vectorize_layer([context_text])

    # Remove padding tokens (0) so the model predicts based on actual words
    mask = input_ids != 0
    input_ids = tf.expand_dims(tf.boolean_mask(input_ids, mask), 0)

    states = None
    generated_tokens = []

    for _ in range(num_generate):
        predictions, states = model(input_ids, states=states, return_state=True)
        predictions = predictions[:, -1, :]

        # Apply temperature scaling to the logits
        # Temperature < 1.0 makes the model more confident/conservative.
        # Temperature > 1.0 makes the distribution flatter, resulting in more diverse/random text.
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[0][0].numpy()

        # Convert ID back to word
        predicted_word = idx2word.get(predicted_id, "")

        # Stop if EOS token is predicted
        if predicted_word == "eos":
            break

        generated_tokens.append(predicted_word)

        # The new input is the predicted ID
        input_ids = tf.expand_dims([predicted_id], 0)

    return " ".join(generated_tokens)

In [ ]:
print("="*50)
print("Welcome to the RNN Chatbot!")
print("Type \"quit\" or \"exit\" to end the conversation.")
print("="*50)

MAX_CONTEXT_TURNS = 12
context_history = []

try:
    while True:
        user_input = input("You: ")
        if user_input.lower() in ["quit", "exit"]:
            print("Bot: Goodbye!")
            break

        context_history.append(user_input)
        if len(context_history) > MAX_CONTEXT_TURNS:
            context_history = context_history[-MAX_CONTEXT_TURNS:]
        current_context = " ".join(context_history)

        # Increased num_generate to allow the model to hit 'eos' naturally
        response = generate_response(model, current_context, num_generate=20, temperature=0.7)

        print(f"Bot: {response}")
        context_history.append(response)
        if len(context_history) > MAX_CONTEXT_TURNS:
            context_history = context_history[-MAX_CONTEXT_TURNS:]

except KeyboardInterrupt:
    print("\nConversation ended by user.")

## Postscript: Beyond LSTMs - The Transformer Era

While LSTMs represent a massive leap over vanilla RNNs by mitigating the vanishing gradient problem, they still suffer from a fundamental architectural limitation: **sequential computation**.

### The Sequential Bottleneck








In an LSTM, to process the 100th word in a sequence, the network must first process words 1 through 99. This inherently sequential nature means that training cannot be easily parallelized across modern GPU hardware. Furthermore, while the cell state ($C_t$) helps preserve long-term memory, in practice, LSTMs still struggle to maintain context over sequences longer than a few hundred tokens.


### Enter the Transformer (2017)


In 2017, researchers at Google published the seminal paper *"Attention Is All You Need"*, introducing the **Transformer** architecture. The Transformer completely discarded recurrence (RNNs/LSTMs) and convolutions, relying entirely on a mechanism called **Self-Attention**.


#### 1. Self-Attention (The Core Mechanism)


Instead of reading text left-to-right, a Transformer processes all words in a sequence simultaneously. To understand the relationships between words, it projects each word's embedding into three distinct vectors:
* **Query (Q):** What this word is looking for (e.g., an adjective looking for its noun).
* **Key (K):** What this word can offer (e.g., a noun announcing its properties).
* **Value (V):** The actual semantic content of this word.

The model computes an "attention score" by taking the dot product of a word's Query with every other word's Key in the sequence. This score determines how much *focus* (or attention) the current word should pay to every other word, regardless of their physical distance in the sentence.


#### 2. Multi-Head Attention


Instead of performing this Q, K, V mapping just once, Transformers do it multiple times in parallel across different "heads." This allows the model to simultaneously capture different types of relationships (e.g., one head might track subject-verb agreement, while another tracks pronoun references).


#### 3. Positional Encodings


Because Transformers process everything at once, they inherently lose the order of the words. To fix this, they add a mathematical "time-stamp" or **Positional Encoding** to each word's embedding before processing. This allows the model to differentiate between "The dog bit the man" and "The man bit the dog" without needing to process them sequentially.


### Modern GPT Models and the LLM Paradigm


Models like **GPT** (Generative Pre-trained Transformer) and ChatGPT utilize a specific variant of this architecture known as a **Decoder-Only Transformer**.
* **Masked Self-Attention:** During training, a mask is applied so that when predicting the next word, the attention mechanism can only look at *past* tokens, not future ones. This perfectly aligns with the autoregressive language modeling task we performed in this lab.
* **Massive Parallelism:** Because the entire sequence is processed at once (with the mask hiding the future), training can be highly parallelized, allowing these models to scale to billions or trillions of parameters on massive datasets.

### The Modern Training Pipeline

Modern chatbots go a step further than the simple language modeling we did today. Their training typically involves:
1. **Pre-training:** Predicting the next word on massive corpora (terabytes of internet text) to learn grammar, facts, and reasoning capabilities.
2. **Supervised Fine-Tuning (SFT):** Training on high-quality instruction-response pairs to learn how to be a helpful assistant rather than just a document-continuer.
3. **Reinforcement Learning from Human Feedback (RLHF):** Using human ratings to fine-tune the model to output responses that are safer, more helpful, and better aligned with human preferences.

While the LSTM we built today is the conceptual ancestor of these systems, the Transformer's ability to parallelize training and perfectly route information across long contexts—combined with massive scale and human-aligned fine-tuning—is what enabled the current era of Generative AI.

## An Example Transformer

In [ ]:
NUM_SAMPLES = 250000
VOCAB_SIZE = 50000
SEQ_LENGTH = 100
BATCH_SIZE = 128
EPOCHS = 10

In [ ]:
# To train a language model, we slice the text into sequences of SEQ_LENGTH + 1
# Example:  [word1, word2, word3, word4]
#   Input:  [word1, word2, word3]
#  Target:  [word2, word3, word4]

def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

vectorized_texts = vectorize_layer(texts)

dataset_seq = tf.data.Dataset.from_tensor_slices(vectorized_texts)
dataset_processed = dataset_seq.map(split_input_target)

dataset_processed = (
    dataset_processed
    .shuffle(10000)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, d_model, max_len=2048):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model)
        self.pos_encoding = tf.keras.layers.Embedding(max_len, d_model)

    def call(self, x):
        length = tf.shape(x)[1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.embedding(x)
        embedded_positions = self.pos_encoding(positions)
        return embedded_tokens + embedded_positions

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="relu"),
            tf.keras.layers.Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training=False):
        # Causal attention mask to prevent looking into the future
        seq_len = tf.shape(inputs)[1]
        causal_mask = tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)

        attn_output = self.att(inputs, inputs, attention_mask=causal_mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

class TransformerLanguageModel(tf.keras.Model):
    def __init__(self, vocab_size, max_len, embed_dim, num_heads, ff_dim, num_layers=1, dropout_rate=0.1):
        super().__init__()
        self.pos_embedding = PositionalEmbedding(vocab_size, embed_dim, max_len)
        self.transformer_blocks = [TransformerBlock(embed_dim, num_heads, ff_dim, dropout_rate) for _ in range(num_layers)]
        self.dense = tf.keras.layers.Dense(vocab_size)

    def call(self, inputs, training=False):
        x = self.pos_embedding(inputs)
        for block in self.transformer_blocks:
            x = block(x, training=training)
        return self.dense(x)

In [ ]:
# Hyperparameters for the Transformer
TRANSFORMER_EMBED_DIM = 256
TRANSFORMER_HEADS = 4
TRANSFORMER_FF_DIM = 512
TRANSFORMER_LAYERS = 2

transformer_model = TransformerLanguageModel(
    vocab_size=len(vocab),
    max_len=SEQ_LENGTH,
    embed_dim=TRANSFORMER_EMBED_DIM,
    num_heads=TRANSFORMER_HEADS,
    ff_dim=TRANSFORMER_FF_DIM,
    num_layers=TRANSFORMER_LAYERS
)

In [ ]:
transformer_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
)

In [ ]:
# print(f"Starting training for Transformer model for {EPOCHS} epochs...")
# transformer_history = transformer_model.fit(dataset_processed, epochs=EPOCHS)

In [ ]:
# import os

# transformer_filename = "reddit_transformer_model.weights.h5"
# local_transformer_path = f"/content/{transformer_filename}"
# drive_model_dir = "/content/drive/My Drive/DATA2000/models/"
# drive_transformer_path = os.path.join(drive_model_dir, transformer_filename)

# os.makedirs(drive_model_dir, exist_ok=True)
# transformer_model.save_weights(local_transformer_path)

# !cp {local_transformer_path} "{drive_transformer_path}"

# print(f"Transformer weights saved to Drive at {drive_transformer_path} successfully!")

In [ ]:
transformer_file_id = "1VGtlc_VhN_C_x64Bu4BBFUkr5N4RyU0Q"
transformer_url = f"https://drive.google.com/uc?id={transformer_file_id}"
local_transformer_path = "/content/reddit_transformer_model.weights.h5"

gdown.download(transformer_url, local_transformer_path, quiet=False)

# Build the model by specifying the expected input shape
transformer_model.build(input_shape=(None, SEQ_LENGTH))

# We load the weights into the model we instantiated earlier
transformer_model.load_weights(local_transformer_path)
print("Transformer weights loaded successfully!")

In [ ]:
def generate_transformer_response(
        model,
        context_text,
        num_generate=50,
        temperature=0.7,
        debug=False):
    input_ids = vectorize_layer([context_text])
    mask = input_ids != 0
    input_ids = tf.expand_dims(tf.boolean_mask(input_ids, mask), 0)

    generated_tokens = []

    if debug:
        print("\n--- DEBUG: Generation Trace ---")
    for i in range(num_generate):
        # Truncate to SEQ_LENGTH if the sequence gets too long
        if tf.shape(input_ids)[1] > SEQ_LENGTH:
            input_ids_cond = input_ids[:, -SEQ_LENGTH:]
        else:
            input_ids_cond = input_ids

        predictions = model(input_ids_cond, training=False)
        # Get the prediction for the last token
        predictions = predictions[:, -1, :] / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[0][0].numpy()

        predicted_word = idx2word.get(predicted_id, "")

        if debug:
            print(f"Step {i}: predicted ID {predicted_id} -> '{predicted_word}'")

        if predicted_word == "eos":
            if debug:
                print("--- Hit EOS token, stopping ---\n")
            break

        generated_tokens.append(predicted_word)

        # Append the predicted ID to the sequence for the next timestep
        predicted_id_tensor = tf.cast(tf.expand_dims([predicted_id], 0), tf.int64)
        input_ids = tf.concat([input_ids, predicted_id_tensor], axis=1)

    if not generated_tokens:
        if debug:
            print("--- No tokens generated! ---\n")

    return " ".join(generated_tokens)

In [ ]:
print("="*50)
print("Welcome to the Transformer Chatbot!")
print("Type \"quit\" or \"exit\" to end the conversation.")
print("="*50)

MAX_CONTEXT_TURNS = 12
transformer_context_history = []

try:
    while True:
        user_input = input("You: ")
        if user_input.lower() in ["quit", "exit"]:
            print("Transformer Bot: Goodbye!")
            break

        transformer_context_history.append(user_input)
        if len(transformer_context_history) > MAX_CONTEXT_TURNS:
            transformer_context_history = transformer_context_history[-MAX_CONTEXT_TURNS:]
        current_context = " ".join(transformer_context_history)

        # Use the transformer-specific generation function
        response = generate_transformer_response(
            transformer_model,
            current_context,
            num_generate=50,
            temperature=0.7)

        print(f"Transformer Bot: {response}")
        transformer_context_history.append(response)
        if len(transformer_context_history) > MAX_CONTEXT_TURNS:
            transformer_context_history = transformer_context_history[-MAX_CONTEXT_TURNS:]

except KeyboardInterrupt:
    print("\nConversation ended by user.")